# Progress Review 2 — Spark Smoke Test

This notebook verifies that JupyterLab can connect to the Spark master container and run a basic Spark job.

Goal:

JupyterLab → Spark master → Spark worker

In [1]:
import os
import pyspark
from pyspark import SparkContext

print("PySpark package version:", pyspark.__version__)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

existing_sc = SparkContext._active_spark_context

if existing_sc is not None:
    try:
        if existing_sc._jvm is None or existing_sc._jsc is None:
            raise RuntimeError("JVM not available – context is dead")

        existing_sc.stop()
        print("Stopped existing SparkContext.")

    except Exception as e:
        print(f"Existing SparkContext is already dead ({e}); clearing reference.")

    finally:
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None

print("SparkContext cleared – safe to create a new session.")

PySpark package version: 4.0.0
SPARK_HOME: /opt/conda/lib/python3.13/site-packages/pyspark
SparkContext cleared – safe to create a new session.


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("aviation-pr2-spark-smoke-test")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "aviation-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4041")
    .config("spark.blockManager.port", "4042")
    .config("spark.ui.port", "4040")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark app ID:", spark.sparkContext.applicationId)
spark

Spark version: 4.0.0
Spark app ID: app-20260425120943-0021


In [3]:
df = spark.createDataFrame(
    [
        ("JFK", 25.5),
        ("LAX", 18.2),
        ("ORD", 30.1),
    ],
    ["airport", "wind_speed_kts"],
)

df.show()

+-------+--------------+
|airport|wind_speed_kts|
+-------+--------------+
|    JFK|          25.5|
|    LAX|          18.2|
|    ORD|          30.1|
+-------+--------------+



In [4]:
df.groupBy().avg("wind_speed_kts").show()

+-------------------+
|avg(wind_speed_kts)|
+-------------------+
| 24.599999999999998|
+-------------------+



In [5]:
spark.stop()
print("Spark stopped cleanly.")

Spark stopped cleanly.
